# SENTINEL on the official harness, with the real Qwen3-8B agent

This notebook runs the organizers' evaluator against **our defense service** while the agent
being defended is the official reference agent — `Qwen/Qwen3-8B` — instead of the deterministic
mock. That is the difference between "our decision logic scores well on scripted plans" and
"our decision logic holds up when a real model is the thing proposing actions".

**Kaggle settings (right-hand panel):**

| setting | value |
|---|---|
| Accelerator | **GPU T4 x2** (two 16 GB cards — 16-bit Qwen3-8B does not fit on one) |
| Internet | **On** (clones the repos, downloads the weights) |
| Persistence | Variables and files: off is fine; results are zipped at the end |

A single T4 or P100 also works if you set `PRECISION = "4bit"` in the config cell.

**What it does, in order:** install → clone the kit and the defense → fetch the weights →
start the defense service → check the attack actually lands with no defense → run the four
evaluations → write the table and the failure list for the report.

**Budget:** expect ~20–40 min per evaluation split, so ~2–3 h for all four, inside Kaggle's
12 h session and 30 h/week GPU quota. Run the first split, look at the clock, then decide.

## 0 · What hardware did we actually get

In [ ]:
import shutil, subprocess, sys
from pathlib import Path

print(sys.version)
try:
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                          "--format=csv,noheader"], capture_output=True, text=True).stdout.strip() or "no GPU")
except FileNotFoundError:
    print("no nvidia-smi: this notebook needs a GPU accelerator")

for path in ("/kaggle/working", "/kaggle/temp", "/tmp"):
    if Path(path).exists():
        free = shutil.disk_usage(path).free / 1e9
        print(f"{path:16s} {free:6.1f} GB free")

## 1 · Configuration

The only cell you should need to edit.

In [ ]:
DEFENSE_REPO = "https://github.com/wissemkooli/sentinel-indabax.git"
DEFENSE_REF  = "main"
KIT_REPO     = "https://github.com/Skan22/Sentinel_Starter_Kit.git"

PRECISION    = "fp16"     # fp16 on 2x T4 (faithful 16-bit) | 4bit or 8bit on a single card
DEFENSE_PORT = 8099

# (split, attacker, attack_mode). The first pair is the headline number; the mutation/adaptive
# pair is the one that shows the defense is not overfitted to a fixed set of injected strings.
RUNS = [
    ("public",     "static",   "static"),
    ("validation", "static",   "static"),
    ("public",     "mutation", "adaptive"),
    ("validation", "mutation", "adaptive"),
]

RUN_BASELINES = False     # also score allow_all / provenance / heuristic_risk under Qwen3-8B
                          # (one extra pass over the public split each -- check your quota first)

import os, shutil
from pathlib import Path

KAGGLE  = Path("/kaggle").exists()
WORK    = Path("/kaggle/working") if KAGGLE else Path.cwd() / "sentinel-qwen3"
SCRATCH = max((Path(p) for p in ("/kaggle/temp", "/tmp", str(WORK)) if Path(p).exists()),
              key=lambda p: shutil.disk_usage(p).free)
KIT      = WORK / "Sentinel_Starter_Kit"
REPO     = WORK / "sentinel-indabax"
RESULTS  = WORK / "results"
ARTIFACTS = WORK / "artifacts"
for d in (WORK, RESULTS, ARTIFACTS):
    d.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(SCRATCH / "hf")
os.environ["SENTINEL_QWEN_PRECISION"] = PRECISION
print(f"work={WORK}\nweights cache={os.environ['HF_HOME']} ({shutil.disk_usage(SCRATCH).free/1e9:.0f} GB free)")

## 2 · Dependencies

Kaggle ships torch; the rest is the kit's runtime plus the defense service's.

In [ ]:
packages = [
    "transformers>=4.51",   # Qwen3 support landed in 4.51
    "accelerate>=0.30",     # device_map sharding across the two T4s
    "typer>=0.12", "rich>=13.7", "httpx>=0.27",
    "pydantic>=2.8,<3", "fastapi>=0.115", "uvicorn>=0.30", "pyyaml>=6.0.2",
    "kagglehub",
]
if PRECISION in ("4bit", "8bit"):
    packages.append("bitsandbytes>=0.43")

spec = " ".join(f'"{p}"' for p in packages)
!pip install -q {spec}

import transformers, torch
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| cuda devices", torch.cuda.device_count())
if torch.cuda.device_count() < 2 and PRECISION in ("fp16", "bf16"):
    print("\nWARNING: one GPU and 16-bit weights will not fit. Set PRECISION = '4bit' and rerun cell 1.")

## 3 · The kit and the defense

In [ ]:
import subprocess, sys

def clone(url, dest, ref=None):
    if dest.exists():
        print(f"{dest.name}: already cloned")
        return
    cmd = ["git", "clone", "--depth", "1"] + (["--branch", ref] if ref else []) + [url, str(dest)]
    subprocess.run(cmd, check=True)

clone(KIT_REPO, KIT)
clone(DEFENSE_REPO, REPO, DEFENSE_REF)

# The kit declares python >=3.12,<3.13 and Kaggle may be on 3.11; nothing in it needs 3.12, so
# import it from source instead of installing the package metadata.
sys.path[:0] = [str(KIT / "src"), str(REPO / "kaggle"), str(REPO), str(REPO / "submission")]
os.environ["SENTINEL_ROOT"] = str(KIT)   # where policies/ and fixtures/ live

for repo in (KIT, REPO):
    head = subprocess.run(["git", "-C", str(repo), "log", "-1", "--format=%h %s"],
                          capture_output=True, text=True).stdout.strip()
    print(f"{repo.name:24s} {head}")

## 4 · Qwen3-8B weights

Tried in order: a Kaggle Model attached as a notebook input, then Kaggle Models, then the
Hugging Face hub. ~16 GB either way, so this is the slow cell on a cold start.

In [ ]:
import json

def looks_like_qwen3_8b(config: Path) -> bool:
    try:
        data = json.loads(config.read_text())
    except Exception:
        return False
    return data.get("model_type", "").startswith("qwen3") and data.get("num_hidden_layers") == 36

def from_inputs():
    root = Path("/kaggle/input")
    if not root.exists():
        return None
    for config in sorted(root.rglob("config.json")):
        if looks_like_qwen3_8b(config):
            return config.parent

def from_kagglehub():
    import kagglehub
    for handle in ("qwen-lm/qwen-3/transformers/8b", "qwen-lm/qwen-3/transformers/qwen3-8b"):
        try:
            return Path(kagglehub.model_download(handle))
        except Exception as exc:
            print(f"  {handle}: {type(exc).__name__}")

def from_huggingface():
    from huggingface_hub import snapshot_download
    return Path(snapshot_download("Qwen/Qwen3-8B"))

WEIGHTS = os.environ.get("SENTINEL_WEIGHTS")
if not WEIGHTS:
    for source in (from_inputs, from_kagglehub, from_huggingface):
        print(f"trying {source.__name__} ...")
        found = source()
        if found:
            WEIGHTS = str(found)
            break
if not WEIGHTS:
    raise SystemExit("no Qwen3-8B weights: attach the Kaggle model as an input, or enable internet")

size = sum(f.stat().st_size for f in Path(WEIGHTS).rglob("*") if f.is_file()) / 1e9
print(f"\nweights: {WEIGHTS}  ({size:.1f} GB)")

## 5 · Start the defense service

The same FastAPI app the submission container runs, on the same `/v1/decision` contract. The
evaluator reaches it over HTTP exactly as the organizers' runner would.

In [ ]:
import time, urllib.request, urllib.error

DEFENSE_URL = f"http://127.0.0.1:{DEFENSE_PORT}"
log_path = RESULTS / "defense-service.log"

def healthz():
    try:
        with urllib.request.urlopen(f"{DEFENSE_URL}/healthz", timeout=2) as response:
            return json.loads(response.read())
    except (urllib.error.URLError, OSError, json.JSONDecodeError):
        return None

if healthz() is None:
    env = dict(os.environ, PYTHONPATH=f"{REPO}:{REPO / 'submission'}")
    defense = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "app.main:app", "--port", str(DEFENSE_PORT), "--log-level", "warning"],
        cwd=str(REPO / "submission"), env=env,
        stdout=log_path.open("w"), stderr=subprocess.STDOUT,
    )
    for _ in range(60):
        if healthz():
            break
        if defense.poll() is not None:
            print(log_path.read_text()[-2000:])
            raise SystemExit("the defense service exited on startup")
        time.sleep(1)

status = healthz()
if not status:
    print(log_path.read_text()[-2000:])
    raise SystemExit("the defense service never became healthy")
print("defense:", status)

## 6 · Does the attack even reach the agent?

The kit's own warning: if an *undefended* run does not report `attack_success=True`, the agent
never opened the injected record and every later number is meaningless. So before scoring
anything, run one attack scenario with `allow_all`, then the same scenario against our defense.

This is also where the 16 GB of weights get loaded — once, and then reused for everything below.

In [ ]:
import qwen_runtime
print(qwen_runtime.apply())
from sentinel.cli import app

SCENARIO = str(KIT / "scenarios/public/finance/finance_false_approval.yaml")

def sentinel(*argv):
    """Invoke the kit's CLI in this process, so the model stays loaded between runs."""
    started = time.time()
    try:
        app(args=list(argv), standalone_mode=False)
    except SystemExit as exc:
        if exc.code:
            raise
    print(f"[{time.time() - started:.0f}s]")

sentinel("run", "--scenario", SCENARIO, "--defense", "allow_all", "--model", WEIGHTS,
         "--artifacts", str(ARTIFACTS))

In [ ]:
sentinel("run", "--scenario", SCENARIO, "--defense-url", DEFENSE_URL, "--model", WEIGHTS,
         "--artifacts", str(ARTIFACTS))

## 7 · The evaluation

Four passes: both splits, static attacker and the adaptive mutation attacker. Each writes a
scorecard JSON into `results/`, and the per-scenario event logs land in `artifacts/` — those
JSONL files are what `sentinel replay` renders, and what the video and the report cite.

In [ ]:
for split, attacker, mode in RUNS:
    name = f"qwen3-8b-{split}-{attacker}-{mode}"
    print(f"\n{'=' * 72}\n{name}\n{'=' * 72}")
    sentinel("eval", split, "--defense-url", DEFENSE_URL, "--model", WEIGHTS,
             "--attacker", attacker, "--attack-mode", mode,
             "--artifacts", str(ARTIFACTS), "--output", str(RESULTS / f"{name}.json"))

## 8 · Baselines under the same agent (optional)

Only meaningful if you have the quota: each is another full pass over the split.

In [ ]:
if RUN_BASELINES:
    for baseline in ("allow_all", "provenance", "heuristic_risk"):
        print(f"\n{'=' * 72}\nbaseline {baseline}\n{'=' * 72}")
        sentinel("eval", "public", "--defense", baseline, "--model", WEIGHTS,
                 "--artifacts", str(ARTIFACTS), "--output", str(RESULTS / f"qwen3-8b-public-{baseline}.json"))
else:
    print("skipped (RUN_BASELINES = False)")

## 9 · Results

The table goes in `docs/OFFICIAL_HARNESS.md`. The failure list underneath it is the part to read
carefully: with a real agent a scenario can end in `model_error` or `max_steps`, which is the
agent failing the task rather than the defense blocking it, and the two must not be conflated.

In [ ]:
import collect_scorecards

cards = collect_scorecards.load(RESULTS)
report = collect_scorecards.report(cards)
(RESULTS / "RESULTS_QWEN3.md").write_text(report + "\n")
print(report)

In [ ]:
# One archive to download from the notebook's Output tab: scorecards, event logs, service log.
archive = shutil.make_archive(str(WORK / "sentinel-qwen3-results"), "zip", root_dir=str(WORK),
                              base_dir=".", logger=None)
print(archive, f"{Path(archive).stat().st_size / 1e6:.1f} MB")

## What to do with this

1. Put the table and the failure list into `docs/OFFICIAL_HARNESS.md`, alongside the mock-model
   numbers rather than replacing them — the gap between the two *is* a result, and says how much
   of the mock's perfect score came from the mock following a reference plan.
2. Declare the runtime in the technical report: precision, device map, that the weights are loaded
   once per process, and that the prompt, tool cards, decoding and step budget are the kit's own.
   `qwen_runtime.apply()` prints exactly that line.
3. Record the video from these artifacts: `sentinel replay artifacts/<group>/<run>.jsonl`.